<a href="https://colab.research.google.com/github/DaoNguyenMinhThy/Machine_Learning/blob/main/Lab07_DanhGiaMoHinh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: thấy accuracy đánh lừa trên dữ liệu lệch lớp.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
print("Kich thuoc bang:", df.shape)
print(df.head())
print()
print("So sinh vien moi lop:")
print(df["bo_hoc"].value_counts().to_string())
print(f"Ty le bo hoc: {df['bo_hoc'].mean():.3f}")
print()

X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
# stratify=y giữ đúng tỷ lệ bỏ học ở cả tập học lẫn tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print(f"Tap kiem tra: {len(y_te)} ban, {y_te.sum()} ban bo hoc")
print()

# Mô hình lười: ai cũng đoán là lớp đông nhất
luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
du_luoi = luoi.predict(X_te)

lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
du_lr = lr.predict(X_te)

for ten, du in [("Mo hinh luoi", du_luoi), ("Hoi quy logistic", du_lr)]:
    acc = accuracy_score(y_te, du)
    tim_ra = ((du == 1) & (y_te == 1)).sum()
    print(f"{ten:17s} accuracy {acc:.4f}, "
          f"tim ra {tim_ra} tren {y_te.sum()} ban bo hoc")

Kich thuoc bang: (1200, 5)
   diem_tb  so_buoi_vang  tin_chi_no  gio_lam_them  bo_hoc
0      8.0             3           0             0       0
1      8.9             5           1             7       0
2      3.3             3           1             4       0
3      6.4             3           0             7       0
4      7.9             3           5            23       0

So sinh vien moi lop:
bo_hoc
0    1081
1     119
Ty le bo hoc: 0.099

Tap kiem tra: 300 ban, 30 ban bo hoc

Mo hinh luoi      accuracy 0.9000, tim ra 0 tren 30 ban bo hoc
Hoi quy logistic  accuracy 0.9467, tim ra 17 tren 30 ban bo hoc


In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: tự đếm bốn ô của ma trận nhầm lẫn, rồi so với thư viện."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
that = y_te.to_numpy()

# Tự đếm: mỗi ô là một cặp (nhãn thật, nhãn đoán)
tp = np.sum((that == 1) & (du == 1))
fp = np.sum((that == 0) & (du == 1))
fn = np.sum((that == 1) & (du == 0))
tn = np.sum((that == 0) & (du == 0))
print("Tu dem:")
print(f" TP = {tp} (doan bo hoc, dung la bo hoc)")
print(f" FP = {fp} (bao dong nham)")
print(f" FN = {fn} (bo sot)")
print(f" TN = {tn} (doan khong bo, dung la khong bo)")
print(f" Tong = {tp + fp + fn + tn}")
print()

m = confusion_matrix(that, du)
print("confusion_matrix cua scikit-learn:")
print(m)
print("Thu tu khi trai ra: TN, FP, FN, TP =", m.ravel())
print()

acc = (tp + tn) / (tp + fp + fn + tn)
print(f"Accuracy tinh tu bon o: ({tp} + {tn}) / {len(that)} = {acc:.4f}")
print(f"So ban bi bao co nguy co: TP + FP = {tp + fp}")

Tu dem:
 TP = 17 (doan bo hoc, dung la bo hoc)
 FP = 3 (bao dong nham)
 FN = 13 (bo sot)
 TN = 267 (doan khong bo, dung la khong bo)
 Tong = 300

confusion_matrix cua scikit-learn:
[[267   3]
 [ 13  17]]
Thu tu khi trai ra: TN, FP, FN, TP = [267   3  13  17]

Accuracy tinh tu bon o: (17 + 267) / 300 = 0.9467
So ban bi bao co nguy co: TP + FP = 20


In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: precision, recall, F1 tính tay và tính bằng thư viện."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, du).ravel()

p = tp / (tp + fp)
r = tp / (tp + fn)
f1 = 2 * p * r / (p + r)
print("Tinh tay tu bon o:")
print(f" Precision = {tp} / ({tp} + {fp}) = {p:.4f}")
print(f" Recall    = {tp} / ({tp} + {fn}) = {r:.4f}")
print(f" F1        = 2 * {p:.4f} * {r:.4f} / ({p:.4f} + {r:.4f})"
      f" = {f1:.4f}")
print()

print("Tinh bang scikit-learn:")
print(f" Precision = {precision_score(y_te, du):.4f}")
print(f" Recall    = {recall_score(y_te, du):.4f}")
print(f" F1        = {f1_score(y_te, du):.4f}")
print()

print("Bao cao day du:")
print(classification_report(y_te, du, digits=4,
                            target_names=["khong bo", "bo hoc"]))

Tinh tay tu bon o:
 Precision = 17 / (17 + 3) = 0.8500
 Recall    = 17 / (17 + 13) = 0.5667
 F1        = 2 * 0.8500 * 0.5667 / (0.8500 + 0.5667) = 0.6800

Tinh bang scikit-learn:
 Precision = 0.8500
 Recall    = 0.5667
 F1        = 0.6800

Bao cao day du:
              precision    recall  f1-score   support

    khong bo     0.9536    0.9889    0.9709       270
      bo hoc     0.8500    0.5667    0.6800        30

    accuracy                         0.9467       300
   macro avg     0.9018    0.7778    0.8255       300
weighted avg     0.9432    0.9467    0.9418       300



In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: đổi ngưỡng, và chọn ngưỡng đúng cách trên tập học."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
xs = lr.predict_proba(X_te)[:, 1]  # xác suất bỏ học của từng bạn

print("Tap kiem tra, doi nguong")
print("nguong so bao TP FP FN precision recall F1")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    du = (xs >= t).astype(int)
    tp = int(((du == 1) & (y_te == 1)).sum())
    fp = int(((du == 1) & (y_te == 0)).sum())
    fn = int(((du == 0) & (y_te == 1)).sum())
    print(f" {t:.1f} {du.sum():4d} {tp:3d} {fp:3d} {fn:3d}"
          f" {precision_score(y_te, du):.4f}"
          f" {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}")
print()

# Chọn ngưỡng đúng cách: chỉ dùng tập học, qua kiểm định chéo.
# Mỗi bạn trong tập học nhận xác suất từ một mô hình chưa từng thấy bạn ấy.
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5,
                           method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1_hoc = [f1_score(y_tr, (xs_hoc >= t).astype(int)) for t in cac_nguong]
t_tot = cac_nguong[int(np.argmax(f1_hoc))]
print(f"Nguong cho F1 cao nhat tren tap hoc: {t_tot}"
      f" (F1 kiem dinh cheo {max(f1_hoc):.4f})")

# Chọn xong mới dùng tập kiểm tra, và chỉ một lần
du = (xs >= t_tot).astype(int)
print(f"Tap kiem tra voi nguong {t_tot}:"
      f" precision {precision_score(y_te, du):.4f},"
      f" recall {recall_score(y_te, du):.4f},"
      f" F1 {f1_score(y_te, du):.4f}")

Tap kiem tra, doi nguong
nguong so bao TP FP FN precision recall F1
 0.5   20  17   3  13 0.8500 0.5667 0.6800
 0.4   21  18   3  12 0.8571 0.6000 0.7059
 0.3   29  21   8   9 0.7241 0.7000 0.7119
 0.2   42  24  18   6 0.5714 0.8000 0.6667
 0.1   61  26  35   4 0.4262 0.8667 0.5714

Nguong cho F1 cao nhat tren tap hoc: 0.25 (F1 kiem dinh cheo 0.6473)
Tap kiem tra voi nguong 0.25: precision 0.6571, recall 0.7667, F1 0.7077


In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: đường ROC, AUC, và so các mô hình đã học trong môn."""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
xs = lr.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]

# Mỗi ngưỡng cho một điểm của đường ROC
fpr, tpr, nguong = roc_curve(y_te, xs)
print("Duong ROC co", len(fpr), "diem. Vai diem dau:")
print(" bao dong nham recall")
for a, b in list(zip(fpr, tpr))[:6]:
    print(f" {a:.4f}        {b:.4f}")
print(f"AUC cua logistic = {roc_auc_score(y_te, xs):.4f}")
print()

# So các mô hình đã học, cùng tập học, cùng tập kiểm tra
mo_hinh = {
    "Logistic": lr,
    "Naive Bayes": GaussianNB(),
    "KNN, k = 15": make_pipeline(StandardScaler(),
                                 KNeighborsClassifier(15)),
    "Cay sau 4": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Rung 200 cay": RandomForestClassifier(200, random_state=0),
    "SVM RBF": make_pipeline(StandardScaler(),
                             SVC(probability=True, random_state=0)),
}
print("Mo hinh        AUC    recall F1 (nguong 0.5)")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    p = mh.predict_proba(X_te)[:, 1]
    du = (p >= 0.5).astype(int)
    print(f" {ten:14s} {roc_auc_score(y_te, p):.4f}"
          f" {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}")

Duong ROC co 28 diem. Vai diem dau:
 bao dong nham recall
 0.0000        0.0000
 0.0000        0.0333
 0.0000        0.4000
 0.0074        0.4000
 0.0074        0.5333
 0.0111        0.5333
AUC cua logistic = 0.9449

Mo hinh        AUC    recall F1 (nguong 0.5)
 Logistic       0.9449 0.5667 0.6800
 Naive Bayes    0.9336 0.3667 0.5116
 KNN, k = 15    0.9274 0.2333 0.3684
 Cay sau 4      0.8803 0.4667 0.5957
 Rung 200 cay   0.9249 0.5667 0.6538
 SVM RBF        0.8996 0.5000 0.6250


In [6]:
# -*- coding: utf-8 -*-
"""Bước 6: kiểm định chéo với nhiều thước đo, và cân bằng lớp khi học."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]

# Chia không phân tầng: số bạn bỏ học trong tập kiểm tra dao động
print("Khong stratify, so ban bo hoc trong tap kiem tra:")
dem = []
for rs in range(10):
    _, _, _, y_te_thu = train_test_split(X, y, test_size=0.25,
                                         random_state=rs)
    dem.append(int(y_te_thu.sum()))
print("", dem)
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
print("Kiem dinh cheo 5 phan tren tap hoc:")
for tm in ["accuracy", "precision", "recall", "f1"]:
    d = cross_val_score(lr, X_tr, y_tr, cv=5, scoring=tm)
    print(f" {tm:10s} {d.mean():.4f}")
print()

# Cân bằng lớp: phạt nặng hơn khi đoán sai lớp hiếm
cb = make_pipeline(StandardScaler(),
                   LogisticRegression(class_weight="balanced"))
print("Tap kiem tra precision recall F1")
for ten, mh in [("Binh thuong", lr), ("Can bang lop", cb)]:
    du = mh.fit(X_tr, y_tr).predict(X_te)
    print(f" {ten:12s} {precision_score(y_te, du):.4f}"
          f" {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}")

Khong stratify, so ban bo hoc trong tap kiem tra:
 [22, 26, 35, 29, 32, 27, 28, 20, 27, 23]

Kiem dinh cheo 5 phan tren tap hoc:
 accuracy   0.9289
 precision  0.7044
 recall     0.4810
 f1         0.5652

Tap kiem tra precision recall F1
 Binh thuong  0.8500 0.5667 0.6800
 Can bang lop 0.4483 0.8667 0.5909


# 8. Bài tập

In [7]:
#Bài tập 1: Tự viết hàm đánh giá
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def danh_gia(y_that, y_doan):
    that = np.asarray(y_that)
    doan = np.asarray(y_doan)

    tp = np.sum((that == 1) & (doan == 1))
    fp = np.sum((that == 0) & (doan == 1))
    fn = np.sum((that == 1) & (doan == 0))
    tn = np.sum((that == 0) & (doan == 0))

    acc = (tp + tn) / len(that)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1


df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

for ten, mh in [("Hoi quy logistic", lr), ("Mo hinh luoi", luoi)]:
    du = mh.predict(X_te)
    acc1, p1, r1, f1_1 = danh_gia(y_te, du)
    acc2 = accuracy_score(y_te, du)
    p2 = precision_score(y_te, du, zero_division=0)
    r2 = recall_score(y_te, du, zero_division=0)
    f1_2 = f1_score(y_te, du, zero_division=0)

    print(f"{ten}:")
    print(f" Tu viet     : accuracy={acc1:.4f}, precision={p1:.4f}, recall={r1:.4f}, F1={f1_1:.4f}")
    print(f" scikit-learn: accuracy={acc2:.4f}, precision={p2:.4f}, recall={r2:.4f}, F1={f1_2:.4f}")
    print()

print("Cau hoi 1: Voi mo hinh luoi, precision co tinh duoc khong?")
print("Tra loi  : Theo toan hoc la KHONG xac dinh (0/0), vi mo hinh luoi luon doan lop 0 nen TP = 0 va FP = 0, lam mau so TP + FP = 0.")
print()
print("Cau hoi 2: Ham cua em xu ly truong hop chia cho 0 the nao?")
print("Tra loi  : Trong ham danh_gia, em kiem tra dieu kien mau so (TP + FP > 0, TP + FN > 0, Precision + Recall > 0) truoc khi chia; neu mau so bang 0 thi tra ve gia tri 0.0 (giong tham so zero_division=0 cua scikit-learn) de tranh loi chia cho 0.")

Hoi quy logistic:
 Tu viet     : accuracy=0.9467, precision=0.8500, recall=0.5667, F1=0.6800
 scikit-learn: accuracy=0.9467, precision=0.8500, recall=0.5667, F1=0.6800

Mo hinh luoi:
 Tu viet     : accuracy=0.9000, precision=0.0000, recall=0.0000, F1=0.0000
 scikit-learn: accuracy=0.9000, precision=0.0000, recall=0.0000, F1=0.0000

Cau hoi 1: Voi mo hinh luoi, precision co tinh duoc khong?
Tra loi  : Theo toan hoc la KHONG xac dinh (0/0), vi mo hinh luoi luon doan lop 0 nen TP = 0 va FP = 0, lam mau so TP + FP = 0.

Cau hoi 2: Ham cua em xu ly truong hop chia cho 0 the nao?
Tra loi  : Trong ham danh_gia, em kiem tra dieu kien mau so (TP + FP > 0, TP + FN > 0, Precision + Recall > 0) truoc khi chia; neu mau so bang 0 thi tra ve gia tri 0.0 (giong tham so zero_division=0 cua scikit-learn) de tranh loi chia cho 0.


In [8]:
#Bài tập 2: Chữ số viết tay
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)

print("Bao cao phan loai 10 chu so:")
print(classification_report(y_te, du_doan, digits=4))

bao_cao = classification_report(y_te, du_doan, output_dict=True)
ds_so = [str(i) for i in range(10)]
so_rec_min = min(ds_so, key=lambda s: bao_cao[s]["recall"])
so_prec_min = min(ds_so, key=lambda s: bao_cao[s]["precision"])

print(f"Cau hoi 1: Chu so nao co recall thap nhat va chu so nao co precision thap nhat?")
print(f"Tra loi  : Chu so {so_rec_min} co recall thap nhat ({bao_cao[so_rec_min]['recall']:.4f}), chu so {so_prec_min} co precision thap nhat ({bao_cao[so_prec_min]['precision']:.4f}).")
print()
print("Cau hoi 2: Vi sao voi bo du lieu nay, macro avg va weighted avg gan nhu bang nhau?")
print("Tra loi  : Vi 10 chu so co so luong anh rat can bang nhau (khoang 44-46 anh moi chu so trong tap kiem tra, support gan bang nhau), nen trung binh cong don gian (macro avg) va trung binh co trong so theo support (weighted avg) cho ket qua gan nhu trung nhau.")

Bao cao phan loai 10 chu so:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        45
           1     0.9787    1.0000    0.9892        46
           2     1.0000    1.0000    1.0000        44
           3     0.9787    1.0000    0.9892        46
           4     1.0000    0.9778    0.9888        45
           5     1.0000    1.0000    1.0000        46
           6     1.0000    1.0000    1.0000        45
           7     1.0000    0.9778    0.9888        45
           8     0.9767    0.9767    0.9767        43
           9     0.9778    0.9778    0.9778        45

    accuracy                         0.9911       450
   macro avg     0.9912    0.9910    0.9911       450
weighted avg     0.9912    0.9911    0.9911       450

Cau hoi 1: Chu so nao co recall thap nhat va chu so nao co precision thap nhat?
Tra loi  : Chu so 8 co recall thap nhat (0.9767), chu so 8 co precision thap nhat (0.9767).

Cau hoi 2: Vi sao voi bo du lieu nay, m

In [9]:
#Bài tập 3: Chọn ngưỡng theo yêu cầu
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]

cac_nguong = np.round(np.arange(0.01, 1.00, 0.01), 2)
nguong_dat = [t for t in cac_nguong if recall_score(y_tr, (xs_hoc >= t).astype(int)) >= 0.8]
t_chon = max(nguong_dat)
rec_cv = recall_score(y_tr, (xs_hoc >= t_chon).astype(int))

print(f"Nguong lon nhat tren tap hoc dat recall >= 0.8: {t_chon:.2f} (recall CV = {rec_cv:.4f})")

lr.fit(X_tr, y_tr)
xs_te = lr.predict_proba(X_te)[:, 1]
du_te = (xs_te >= t_chon).astype(int)

rec_te = recall_score(y_te, du_te)
prec_te = precision_score(y_te, du_te)
so_bao = int(du_te.sum())

print(f"Ket qua tap kiem tra voi nguong {t_chon:.2f}:")
print(f" Recall = {rec_te:.4f}, Precision = {prec_te:.4f}, So ban bi bao = {so_bao} tren {len(y_te)}")
print()

dat_yeu_cau = "CO" if rec_te >= 0.8 else "KHONG"
print("Cau hoi: Ap nguong do cho tap kiem tra, recall co dat 0.8 khong? Precision va so ban bi bao bang bao nhieu?")
print(f"Tra loi: Recall tren tap kiem tra dat {rec_te:.4f} ({dat_yeu_cau} dat muc 0.8), precision dat {prec_te:.4f} va co {so_bao} ban bi bao co nguy co bo hoc.")

Nguong lon nhat tren tap hoc dat recall >= 0.8: 0.17 (recall CV = 0.8090)
Ket qua tap kiem tra voi nguong 0.17:
 Recall = 0.8000, Precision = 0.5106, So ban bi bao = 47 tren 300

Cau hoi: Ap nguong do cho tap kiem tra, recall co dat 0.8 khong? Precision va so ban bi bao bang bao nhieu?
Tra loi: Recall tren tap kiem tra dat 0.8000 (CO dat muc 0.8), precision dat 0.5106 va co 47 ban bi bao co nguy co bo hoc.


In [10]:
#Bài tập 4: Tự vẽ đường ROC
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
rf = RandomForestClassifier(200, random_state=0).fit(X_tr, y_tr)

xs_lr = lr.predict_proba(X_te)[:, 1]
xs_rf = rf.predict_proba(X_te)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_te, xs_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_te, xs_rf)

auc_lr = roc_auc_score(y_te, xs_lr)
auc_rf = roc_auc_score(y_te, xs_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f"Hoi quy logistic (AUC = {auc_lr:.4f})", linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f"Rung ngau nhien (AUC = {auc_rf:.4f})", linewidth=2)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Doan ngau nhien (AUC = 0.5000)")
plt.xlabel("Ty le bao dong nham (FPR)")
plt.ylabel("Recall (TPR)")
plt.title("Duong ROC so sanh Logistic va Rung ngau nhien")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("roc.png", dpi=150)
plt.close()

print(f"AUC Logistic = {auc_lr:.4f}, AUC Rung ngau nhien = {auc_rf:.4f}")
print("Da luu hinh roc.png")
print()
print("Cau hoi 1: Duong nao om sat goc tren ben trai hon?")
print("Tra loi  : Duong ROC cua Hoi quy logistic om sat goc tren ben trai hon (AUC = 0.9449 cao hon 0.9249 cua Rung ngau nhien).")
print()
print("Cau hoi 2: Co doan nao hai duong cat nhau khong?")
print("Tra loi  : Co, hai duong co cat nhau o mot vai doan nho (khoang vung FPR thap duoi 0.2 va doan FPR quanh 0.7), nhung nhin chung duong Logistic nam phia tren duong Rung ngau nhien o phan lon khoang nguong.")

AUC Logistic = 0.9449, AUC Rung ngau nhien = 0.9249
Da luu hinh roc.png

Cau hoi 1: Duong nao om sat goc tren ben trai hon?
Tra loi  : Duong ROC cua Hoi quy logistic om sat goc tren ben trai hon (AUC = 0.9449 cao hon 0.9249 cua Rung ngau nhien).

Cau hoi 2: Co doan nao hai duong cat nhau khong?
Tra loi  : Co, hai duong co cat nhau o mot vai doan nho (khoang vung FPR thap duoi 0.2 va doan FPR quanh 0.7), nhung nhin chung duong Logistic nam phia tren duong Rung ngau nhien o phan lon khoang nguong.


In [11]:
#Bài tập 5: Cân bằng lớp cho mô hình khác
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = {
    "RF binh thuong": RandomForestClassifier(200, random_state=0),
    "RF can bang"   : RandomForestClassifier(200, class_weight="balanced", random_state=0),
    "SVC binh thuong": make_pipeline(StandardScaler(), SVC(random_state=0)),
    "SVC can bang"   : make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=0)),
}

print("Mo hinh           precision  recall      F1")
ket_qua = {}
for ten, mh in cac_mo_hinh.items():
    du = mh.fit(X_tr, y_tr).predict(X_te)
    p = precision_score(y_te, du)
    r = recall_score(y_te, du)
    f1 = f1_score(y_te, du)
    ket_qua[ten] = (p, r, f1)
    print(f"{ten:16s}  {p:9.4f}  {r:6.4f}  {f1:6.4f}")
print()

thay_doi_rf = abs(ket_qua["RF can bang"][1] - ket_qua["RF binh thuong"][1])
thay_doi_svc = abs(ket_qua["SVC can bang"][1] - ket_qua["SVC binh thuong"][1])
mh_doi_nhieu = "SVC" if thay_doi_svc > thay_doi_rf else "RandomForestClassifier"

print("Cau hoi: So precision, recall, F1 truoc va sau khi can bang. Mo hinh nao thay doi nhieu nhat?")
print(f"Tra loi: Khi them class_weight='balanced', {mh_doi_nhieu} thay doi nhieu nhat (recall va precision bien dong manh hon han so voi mo hinh con lai, vi trong so phat lop hiem lam dich chuyen manh ranh gioi quyet dinh cua mo hinh).")

Mo hinh           precision  recall      F1
RF binh thuong       0.7727  0.5667  0.6538
RF can bang          0.7500  0.4000  0.5217
SVC binh thuong      0.8824  0.5000  0.6383
SVC can bang         0.4464  0.8333  0.5814

Cau hoi: So precision, recall, F1 truoc va sau khi can bang. Mo hinh nao thay doi nhieu nhat?
Tra loi: Khi them class_weight='balanced', SVC thay doi nhieu nhat (recall va precision bien dong manh hon han so voi mo hinh con lai, vi trong so phat lop hiem lam dich chuyen manh ranh gioi quyet dinh cua mo hinh).


In [12]:
#Bài tập 6: MAE và RMSE với một lần đoán hỏng
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression().fit(X_tr, y_tr)
du_goc = lr.predict(X_te)

mae_1 = mean_absolute_error(y_te, du_goc)
rmse_1 = root_mean_squared_error(y_te, du_goc)
r2_1 = r2_score(y_te, du_goc)

du_hong = du_goc.copy()
du_hong[0] += 5.0

mae_2 = mean_absolute_error(y_te, du_hong)
rmse_2 = root_mean_squared_error(y_te, du_hong)
r2_2 = r2_score(y_te, du_hong)

print("Truong hop        MAE     RMSE      R2")
print(f"Ban dau        {mae_1:6.4f}   {rmse_1:6.4f}  {r2_1:6.4f}")
print(f"Doan hong +5   {mae_2:6.4f}   {rmse_2:6.4f}  {r2_2:6.4f}")
print(f"Chenh lech     {abs(mae_2 - mae_1):6.4f}   {abs(rmse_2 - rmse_1):6.4f}  {abs(r2_2 - r2_1):6.4f}")
print()

print("Cau hoi 1: Con so nao thay doi nhieu nhat?")
print("Tra loi  : RMSE (va R2 tuong ung) thay doi manh hon nhieu so voi MAE khi chi mot can bi doan sai lech 5 ty.")
print()
print("Cau hoi 2: Giai thich vi sao RMSE phat nang sai so lon hon MAE?")
print("Tra loi  : Vi MAE lay trung binh tri tuyet doi cua sai so (sai them 5 ty chi tang tong sai so them 5), trong khi RMSE binh phuong tung sai so truoc khi lay trung binh va khai can (sai lech 5 ty bi binh phuong thanh 25), nen mot sai so ngoai lai lon se keo RMSE tang vot.")

Truong hop        MAE     RMSE      R2
Ban dau        0.2578   0.3730  0.9622
Doan hong +5   0.6745   1.5505  0.3475
Chenh lech     0.4167   1.1775  0.6148

Cau hoi 1: Con so nao thay doi nhieu nhat?
Tra loi  : RMSE (va R2 tuong ung) thay doi manh hon nhieu so voi MAE khi chi mot can bi doan sai lech 5 ty.

Cau hoi 2: Giai thich vi sao RMSE phat nang sai so lon hon MAE?
Tra loi  : Vi MAE lay trung binh tri tuyet doi cua sai so (sai them 5 ty chi tang tong sai so them 5), trong khi RMSE binh phuong tung sai so truoc khi lay trung binh va khai can (sai lech 5 ty bi binh phuong thanh 25), nen mot sai so ngoai lai lon se keo RMSE tang vot.
